# Notebook 09 - Uncertainty Quantification (Second version)

City-agnostic improved workflow for **Copenhagen**. This version keeps the original March 2026 `NB09` untouched and adds three targeted improvements:
- spatially explicit daily raster modifiers for trees and waste heat
- event-level warning-day logic for EWS
- EWS CBA uncertainty outputs alongside the impact uncertainty outputs

All new outputs are written to `outputs/copenhagen/tables/uncertainty_improved_fast/` so the original `NB09` outputs remain unchanged.


In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault('CITY', 'copenhagen')
os.environ.setdefault('NB09_N', '512')
os.environ.setdefault('NB09_SEED', '42')
os.environ.setdefault('NB09_MAKE_FIGURES', '0')
os.environ.setdefault('HAZARD_TRACK', 'extreme')

def _find_repo_root():
    env = os.environ.get("URBAN_HEAT_ROOT")
    if env:
        root = Path(env).expanduser().resolve()
        if (root / "cityheat").is_dir() and (root / "configs").is_dir():
            return root
    start = Path.cwd().resolve()
    for cand in [start, *start.parents]:
        if (cand / "cityheat").is_dir() and (cand / "configs").is_dir():
            return cand
    raise RuntimeError(
        "Could not locate repo root. Run the notebook from inside the repo "
        "or set URBAN_HEAT_ROOT."
    )

ROOT = _find_repo_root()
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'outputs' / '.mpl'))
os.environ.setdefault('XDG_CACHE_HOME', str(ROOT / 'outputs' / '.cache'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)
Path(os.environ['XDG_CACHE_HOME']).mkdir(parents=True, exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('ROOT:', ROOT)
print('CITY:', os.environ['CITY'])
print('NB09_N:', os.environ['NB09_N'])
print('NB09_SEED:', os.environ['NB09_SEED'])


In [ ]:
from cityheat.nb09_improved_fast import run_nb09_improved_fast

paths = run_nb09_improved_fast()
paths


In [ ]:
import pandas as pd

samples = pd.read_csv(paths['samples'])
samples[[
    'year', 'aai_agg', 'annual_deaths', 'sample_warning_days',
    'sample_threshold_deaths_per_day', 'ews_pv_cost_25y',
    'ews_net_avoided_deaths_25y_cum', 'ews_cost_per_net_death_25y_cum'
]].head(10)

In [ ]:
impact_summary = pd.read_csv(paths['impact'])
cba_summary = pd.read_csv(paths['cba'])

display(impact_summary.describe())
display(cba_summary.describe())